In [53]:
import pandas as pd
import numpy as np
import pickle

from pathlib import Path
from scipy.sparse import save_npz, load_npz

print("WORKING DIRECTORY:", os.getcwd())
print("PYTHON:", sys.executable)
print("PROJECT CONTENTS:", os.listdir(".."))

WORKING DIRECTORY: /Users/krrishmujoo/Projects/AI-Recomender-project/notebooks
PYTHON: /Users/krrishmujoo/Projects/AI-Recomender-project/.venv/bin/python
PROJECT CONTENTS: ['artifacts', '.DS_Store', '.venv', 'data', 'notebooks']


In [54]:
print("notebook works")

notebook works


In [55]:
events = pd.read_csv("../data/events.csv")

In [56]:
events.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [57]:
events.shape

(2756101, 5)

In [58]:
events_clean = events.drop_duplicates()

In [59]:
events_clean.shape

(2755641, 5)

In [60]:
events_clean["datetime"] = pd.to_datetime(
    events_clean["timestamp"],
    unit="ms"
)

In [61]:
events_clean[["timestamp", "datetime"]].head()

,timestamp,datetime
0,1433221332117,2015-06-02 05:02:12.117
1,1433224214164,2015-06-02 05:50:14.164
2,1433221999827,2015-06-02 05:13:19.827
3,1433221955914,2015-06-02 05:12:35.914
4,1433221337106,2015-06-02 05:02:17.106


In [62]:
train_data = events_clean[
    events_clean["datetime"] < "2015-09-01"
]

test_data = events_clean[
    events_clean["datetime"] >= "2015-09-01"
]

In [63]:
train_data.shape, test_data.shape

((2451983, 6), (303658, 6))

In [64]:
train_users = set(train_data["visitorid"].unique())
test_users = set(test_data["visitorid"].unique())

returning_users = train_users & test_users

In [65]:
len(train_users), len(test_users), len(returning_users)

(1250856, 173728, 17004)

In [66]:
returning_user_history = (
    train_data[train_data["visitorid"].isin(returning_users)]
    .groupby("visitorid")
    .size()
)

In [67]:
returning_user_history.describe()

count    17004.000000
mean         7.048694
std         72.268954
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max       6683.000000
dtype: float64

In [68]:
eligible_users = returning_user_history[
    returning_user_history >= 3
].index

In [69]:
len(eligible_users)

6172

In [70]:
test_eligible = test_data[
    test_data["visitorid"].isin(eligible_users)
]

train_transactions = train_data[
    train_data["event"] == "transaction"
]

popular_items = (
    train_transactions["itemid"]
    .value_counts()
    .index
    .tolist()
)

In [71]:
item_props_sample = pd.read_csv(
    "../data/item_properties_part1.csv",
    nrows=10
)

item_props_sample

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513
5,1436065200000,285026,available,0
6,1434250800000,89534,213,1121373
7,1431831600000,264312,6,319724
8,1433646000000,229370,202,1330310
9,1434250800000,98113,451,1141052 n48.000


In [72]:
item_props_sample.columns

Index(['timestamp', 'itemid', 'property', 'value'], dtype='str')

In [73]:
item_props_sample["property"].value_counts()

property
categoryid    1
888           1
400           1
790           1
917           1
available     1
213           1
6             1
202           1
451           1
Name: count, dtype: int64

In [74]:
item_props_sample = pd.read_csv(
    "../data/item_properties_part1.csv",
    nrows=100000
)

In [75]:
item_props_sample["property"].value_counts().head(15)

property
888           14783
790            8903
available      7338
categoryid     3878
6              3164
283            2926
776            2910
364            2385
678            2351
202            2264
764            2094
112            2045
917            2041
839            2023
159            1982
Name: count, dtype: int64

In [76]:
item_props_sample["property"].nunique()

838

In [77]:
category_rows = item_props_sample[
    item_props_sample["property"] == "categoryid"
]

In [78]:
category_rows.head()

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
140,1432436400000,281245,categoryid,1277
151,1435460400000,35575,categoryid,1059
189,1437274800000,8313,categoryid,1147
197,1437879600000,55102,categoryid,47


In [79]:
category_rows["itemid"].nunique(), category_rows["value"].nunique()

(3878, 606)

In [80]:
category_tree = pd.read_csv("../data/category_tree.csv")

In [81]:
category_tree.head(10)

,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0
5,231,NaN
6,542,378.0
7,1146,542.0
8,1140,542.0
9,1479,1537.0


In [82]:
category_tree.shape

(1669, 2)

In [83]:
category_rows = item_props_sample[
    item_props_sample["property"] == "categoryid"
]

In [84]:
category_rows = category_rows.copy()
category_rows["categoryid"] = pd.to_numeric(
    category_rows["value"],
    errors="coerce"
)

In [85]:
example_item = category_rows.iloc[0]["itemid"]
example_category = category_rows.iloc[0]["categoryid"]

example_item, example_category

(np.int64(460429), np.int64(1338))

In [86]:
same_category_items = category_rows[
    (category_rows["categoryid"] == example_category) &
    (category_rows["itemid"] != example_item)
]["itemid"].unique()

same_category_items[:10]

array([187386, 358218])

In [87]:
example_parent = category_tree.loc[
    category_tree["categoryid"] == example_category,
    "parentid"
].iloc[0]

example_category, example_parent

(np.int64(1338), np.float64(1278.0))

In [88]:
sibling_categories = category_tree.loc[
    (category_tree["parentid"] == example_parent) &
    (category_tree["categoryid"] != example_category),
    "categoryid"
].tolist()

sibling_categories[:10], len(sibling_categories)

([578, 1374], 2)

In [89]:
sibling_items = category_rows[
    category_rows["categoryid"].isin(sibling_categories)
]["itemid"].unique()

sibling_items[:10], len(sibling_items)

(array([465310, 384309,  98909]), 3)

In [90]:
recommendations = []

for item in same_category_items:
    recommendations.append({
        "itemid": item,
        "score": 2,
        "reason": "same category"
    })

for item in sibling_items:
    recommendations.append({
        "itemid": item,
        "score": 1,
        "reason": "sibling category"
    })

recommendations_df = pd.DataFrame(recommendations)

recommendations_df.sort_values(
    by="score",
    ascending=False
)

,itemid,score,reason
0,187386,2,same category
1,358218,2,same category
2,465310,1,sibling category
3,384309,1,sibling category
4,98909,1,sibling category


In [91]:
def generate_category_candidates(item_id, category_rows, category_tree):
    item_match = category_rows[category_rows["itemid"] == item_id]

    if item_match.empty:
        return pd.DataFrame(columns=["itemid", "score", "reason"])

    item_category = item_match.iloc[0]["categoryid"]

    parent_match = category_tree[
        category_tree["categoryid"] == item_category
    ]

    if parent_match.empty:
        parent_id = None
    else:
        parent_id = parent_match.iloc[0]["parentid"]

    same_category_items = category_rows[
        (category_rows["categoryid"] == item_category) &
        (category_rows["itemid"] != item_id)
    ]["itemid"].unique()

    recommendations = []

    for candidate in same_category_items:
        recommendations.append({
            "itemid": candidate,
            "score": 2,
            "reason": "same category"
        })

    if pd.notna(parent_id):
        sibling_categories = category_tree.loc[
            (category_tree["parentid"] == parent_id) &
            (category_tree["categoryid"] != item_category),
            "categoryid"
        ].tolist()

        sibling_items = category_rows[
            category_rows["categoryid"].isin(sibling_categories)
        ]["itemid"].unique()

        for candidate in sibling_items:
            recommendations.append({
                "itemid": candidate,
                "score": 1,
                "reason": "sibling category"
            })

    recommendations_df = pd.DataFrame(recommendations)

    if recommendations_df.empty:
        return recommendations_df

    return (
        recommendations_df
        .drop_duplicates(subset="itemid")
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )

In [92]:
generate_category_candidates(
    460429,
    category_rows,
    category_tree
)

,itemid,score,reason
0,187386,2,same category
1,358218,2,same category
2,465310,1,sibling category
3,384309,1,sibling category
4,98909,1,sibling category


In [93]:
purchase_counts = (
    train_transactions["itemid"]
    .value_counts()
)

NameError: name 'train_transactions' is not defined

In [ ]:
train_transactions = train_data[
    train_data["event"] == "transaction"
]

In [ ]:
train_transactions.shape

(20088, 6)

In [ ]:
purchase_counts = (
    train_transactions["itemid"]
    .value_counts()
)

In [ ]:
purchase_counts.head(10)

itemid
461686    109
119736     91
213834     67
445351     43
48030      41
7943       40
420960     37
17478      37
248455     37
312728     34
Name: count, dtype: int64

In [ ]:
candidates = generate_category_candidates(
    460429,
    category_rows,
    category_tree
)

In [ ]:
candidates = candidates.rename(
    columns={"score": "category_score"}
)

In [ ]:
candidates["purchase_count"] = (
    candidates["itemid"]
    .map(purchase_counts)
    .fillna(0)
    .astype(int)
)

In [ ]:
ranked_candidates = (
    candidates
    .sort_values(
        by=["category_score", "purchase_count"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_candidates

,itemid,category_score,reason,purchase_count
0,187386,2,same category,0
1,358218,2,same category,0
2,465310,1,sibling category,0
3,384309,1,sibling category,0
4,98909,1,sibling category,0


In [ ]:
interaction_counts = (
    train_data["itemid"]
    .value_counts()
)

NameError: name 'train_data' is not defined

In [ ]:
candidates["interaction_count"] = (
    candidates["itemid"]
    .map(interaction_counts)
    .fillna(0)
    .astype(int)
)

In [ ]:
ranked_candidates = (
    candidates
    .sort_values(
        by=["category_score", "purchase_count", "interaction_count"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

ranked_candidates

,itemid,category_score,reason,purchase_count,interaction_count
0,187386,2,same category,0,6
1,358218,2,same category,0,2
2,98909,1,sibling category,0,7
3,465310,1,sibling category,0,0
4,384309,1,sibling category,0,0


In [ ]:
property_counts = (
    item_props_sample["property"]
    .value_counts()
)

property_counts.head(20)

property
888           14783
790            8903
available      7338
categoryid     3878
6              3164
283            2926
776            2910
364            2385
678            2351
202            2264
764            2094
112            2045
917            2041
839            2023
159            1982
227            1666
698            1401
451            1249
663            1189
962            1183
Name: count, dtype: int64

In [ ]:
top_properties = property_counts.head(10).index.tolist()

top_properties

['888',
 '790',
 'available',
 'categoryid',
 '6',
 '283',
 '776',
 '364',
 '678',
 '202']

In [ ]:
for prop in top_properties:
    print("\nPROPERTY:", prop)
    print(
        item_props_sample[
            item_props_sample["property"] == prop
        ]["value"].head(8).tolist()
    )


PROPERTY: 888
['1116713 960601 n277.200', '1038400 45956 n504.000', '1292080', '150169 176547 824301 24474 293011 1240134', '599031', '883447 726612', '1051803 1227205 n712862568.000 992862 951748 690896 n30.000 1135780 1284577 n187.200 1284577 n16392.000 628176 n9216.000 134030 1284577 n72.000 1187104 1284577 n12.000 1175087 n12.000 1072114 1284577 1186729 681805 1284577 618514 980588 927741 32766 1284577 1141052 n48.000 288796 1284577 842796 1154859', '1024724 220869']

PROPERTY: 790
['n15360.000', 'n21000.000', 'n5400.000', 'n39588.000', 'n10320.000', 'n185280.000', 'n11160.000', 'n15588.000']

PROPERTY: available
['0', '0', '0', '1', '0', '0', '0', '0']

PROPERTY: categoryid
['1338', '1277', '1059', '1147', '47', '619', '1228', '1277']

PROPERTY: 6
['319724', '1214748 1186610', '377560', '153715 484436', '452320 853231', '1115994', '1283144 353870', '1283144 353870']

PROPERTY: 283
['132734 242183 797785', '1319279 920988 639039', '554317 820477 938947', '882006 768453 1113872 123

In [ ]:
item_props_sample["feature_token"] = (
    item_props_sample["property"].astype(str)
    + "_"
    + item_props_sample["value"].astype(str)
)

In [ ]:
item_props_sample[
    ["itemid", "property", "value", "feature_token"]
].head(10)

,itemid,property,value,feature_token
0,460429,categoryid,1338,categoryid_1338
1,206783,888,1116713 960601 n277.200,888_1116713 960601 n277.200
2,395014,400,n552.000 639502 n720.000 424566,400_n552.000 639502 n720.000 424566
3,59481,790,n15360.000,790_n15360.000
4,156781,917,828513,917_828513
5,285026,available,0,available_0
6,89534,213,1121373,213_1121373
7,264312,6,319724,6_319724
8,229370,202,1330310,202_1330310
9,98113,451,1141052 n48.000,451_1141052 n48.000


In [ ]:
item_profiles = (
    item_props_sample
    .groupby("itemid")["feature_token"]
    .apply(lambda tokens: " ".join(tokens))
)

In [ ]:
item_profiles.head()

itemid
22                        112_679677
28                        790_n0.000
35                        112_679677
42    available_1 917_416868 1107526
52                    categoryid_512
Name: feature_token, dtype: str

In [ ]:
def make_feature_tokens(row):
    prop = str(row["property"])
    values = str(row["value"]).split()

    return [f"{prop}_{value}" for value in values]

In [ ]:
item_props_sample["feature_tokens"] = item_props_sample.apply(
    make_feature_tokens,
    axis=1
)

In [ ]:
item_props_sample[
    ["itemid", "property", "value", "feature_tokens"]
].head(10)

,itemid,property,value,feature_tokens
0,460429,categoryid,1338,[categoryid_1338]
1,206783,888,1116713 960601 n277.200,"[888_1116713, 888_960601, 888_n277.200]"
2,395014,400,n552.000 639502 n720.000 424566,"[400_n552.000, 400_639502, 400_n720.000, 400_4..."
3,59481,790,n15360.000,[790_n15360.000]
4,156781,917,828513,[917_828513]
5,285026,available,0,[available_0]
6,89534,213,1121373,[213_1121373]
7,264312,6,319724,[6_319724]
8,229370,202,1330310,[202_1330310]
9,98113,451,1141052 n48.000,"[451_1141052, 451_n48.000]"


In [ ]:
item_token_profiles = (
    item_props_sample
    .groupby("itemid")["feature_tokens"]
    .sum()
)

In [ ]:
item_token_profiles.head()

itemid
22                              [112_679677]
28                              [790_n0.000]
35                              [112_679677]
42    [available_1, 917_416868, 917_1107526]
52                          [categoryid_512]
Name: feature_tokens, dtype: object

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
mlb = MultiLabelBinarizer(sparse_output=True)

In [ ]:
item_feature_matrix = mlb.fit_transform(
    item_token_profiles
)

In [ ]:
item_feature_matrix.shape

(87595, 57478)

In [ ]:
len(mlb.classes_)

57478

In [ ]:
item_ids = item_token_profiles.index.to_numpy()

In [ ]:
item_ids[0]

np.int64(22)

In [ ]:
target_item = 460429

In [ ]:
target_index = np.where(item_ids == target_item)[0][0]

target_index

np.int64(86378)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarities = cosine_similarity(
    item_feature_matrix[target_index],
    item_feature_matrix
).flatten()

In [ ]:
similarities.shape

(87595,)

In [ ]:
top_indices = similarities.argsort()[::-1]

In [ ]:
top_indices = top_indices[top_indices != target_index]

In [ ]:
top_10_indices = top_indices[:10]

In [ ]:
top_10_items = item_ids[top_10_indices]

In [ ]:
top_10_scores = similarities[top_10_indices]

In [ ]:
top_10_similar = pd.DataFrame({
    "itemid": top_10_items,
    "similarity": top_10_scores
})

top_10_similar

,itemid,similarity
0,187386,1.0
1,358218,1.0
2,155346,0.0
3,155312,0.0
4,155319,0.0
5,155322,0.0
6,155325,0.0
7,155328,0.0
8,155333,0.0
9,155338,0.0


In [ ]:
item_token_profiles.loc[target_item]

['categoryid_1338']

In [ ]:
len(item_token_profiles.loc[target_item])

1

In [ ]:
item_token_profiles.loc[187386]

['categoryid_1338']

In [ ]:
item_token_profiles.loc[358218]

['categoryid_1338']

In [ ]:
item_props_1 = pd.read_csv("../data/item_properties_part1.csv")
item_props_2 = pd.read_csv("../data/item_properties_part2.csv")

In [ ]:
item_props_full = pd.concat(
    [item_props_1, item_props_2],
    ignore_index=True
)

In [ ]:
item_props_full.shape

(20275902, 4)

In [ ]:
item_props_latest = (
    item_props_full
    .sort_values("timestamp")
    .drop_duplicates(
        subset=["itemid", "property"],
        keep="last"
    )
)

In [ ]:
item_props_latest.shape

(12003814, 4)

In [ ]:
item_props_latest.head()

,timestamp,itemid,property,value
5668945,1431226800000,422842,480,1133979
11314219,1431226800000,310185,776,103591
15170322,1431226800000,110973,112,679677
15170323,1431226800000,179597,available,0
11314209,1431226800000,138592,764,1285872


In [ ]:
item_props_latest["itemid"].nunique()

417053

In [ ]:
item_props_latest["property"].nunique()

1104

In [ ]:
tokens_full = item_props_latest[
    ["itemid", "property", "value"]
].copy()

tokens_full["value_token"] = tokens_full["value"].astype(str).str.split()

In [ ]:
tokens_full = tokens_full.explode("value_token")

In [ ]:
tokens_full["feature_token"] = (
    tokens_full["property"].astype(str)
    + "_"
    + tokens_full["value_token"].astype(str)
)

In [ ]:
tokens_full.head(10)

,itemid,property,value,value_token,feature_token
5668945,422842,480,1133979,1133979,480_1133979
11314219,310185,776,103591,103591,776_103591
15170322,110973,112,679677,679677,112_679677
15170323,179597,available,0,0,available_0
11314209,138592,764,1285872,1285872,764_1285872
5668941,216269,364,336749,336749,364_336749
5668940,299944,764,1285872,1285872,764_1285872
5668939,146103,112,679677,679677,112_679677
16420879,319462,480,1133979,1133979,480_1133979
5668934,431201,283,628374 827388 738621 628374 372324 703408 1182...,628374,283_628374


In [ ]:
tokens_full.shape

(26221597, 5)

In [ ]:
tokens_full["feature_token"].nunique()

2111617

In [ ]:
tokens_full["itemid"].nunique()

417053

In [ ]:
item_feature_pairs = (
    tokens_full[["itemid", "feature_token"]]
    .drop_duplicates()
)

In [ ]:
feature_counts = item_feature_pairs["feature_token"].value_counts()

In [ ]:
feature_counts.describe()

count    2.111617e+06
mean     1.171998e+01
std      6.598753e+02
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      4.170530e+05
Name: count, dtype: float64

In [ ]:
(feature_counts == 1).sum()

np.int64(1740929)

In [ ]:
for threshold in [2, 3, 5, 10, 20, 50, 100]:
    kept = (feature_counts >= threshold).sum()
    print(threshold, kept)

2 370688
3 248344
5 161379
10 92098
20 56829
50 31063
100 19629


In [ ]:
useful_features = feature_counts[
    feature_counts >= 5
].index

In [ ]:
filtered_pairs = item_feature_pairs[
    item_feature_pairs["feature_token"].isin(useful_features)
]

In [ ]:
filtered_pairs.shape

(22468165, 2)

In [ ]:
filtered_pairs["itemid"].nunique()

417053

In [ ]:
from scipy.sparse import csr_matrix

In [ ]:
item_codes, item_ids_full = pd.factorize(filtered_pairs["itemid"])
feature_codes, feature_names = pd.factorize(filtered_pairs["feature_token"])

In [ ]:
import numpy as np

data = np.ones(len(filtered_pairs), dtype=np.uint8)

full_item_feature_matrix = csr_matrix(
    (
        data,
        (item_codes, feature_codes)
    ),
    shape=(len(item_ids_full), len(feature_names))
)

In [ ]:
full_item_feature_matrix.shape

(417053, 161379)

In [ ]:
full_item_feature_matrix.nnz

22468165

In [ ]:
target_item = 460429

In [ ]:
target_index_full = np.where(item_ids_full == target_item)[0][0]
target_index_full

np.int64(179432)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

full_similarities = cosine_similarity(
    full_item_feature_matrix[target_index_full],
    full_item_feature_matrix
).flatten()

In [ ]:
full_similarities.shape

(417053,)

In [ ]:
top_indices_full = full_similarities.argsort()[::-1]

In [ ]:
top_indices_full = top_indices_full[
    top_indices_full != target_index_full
]

In [ ]:
top_10_indices_full = top_indices_full[:10]

In [ ]:
top_10_items_full = item_ids_full[top_10_indices_full]

In [ ]:
top_10_scores_full = full_similarities[top_10_indices_full]

In [ ]:
top_10_similar_full = pd.DataFrame({
    "itemid": top_10_items_full,
    "similarity": top_10_scores_full
})

top_10_similar_full

,itemid,similarity
0,100656,0.916667
1,440711,0.893819
2,238891,0.875000
3,367270,0.875000
4,402554,0.875000
5,283316,0.833333
6,126935,0.833333
7,318140,0.808694
8,78617,0.808694
9,47106,0.808694


In [ ]:
target_features = set(
    filtered_pairs.loc[
        filtered_pairs["itemid"] == 460429,
        "feature_token"
    ]
)

match_features = set(
    filtered_pairs.loc[
        filtered_pairs["itemid"] == 100656,
        "feature_token"
    ]
)

shared_features = target_features & match_features

len(target_features), len(match_features), len(shared_features)

(24, 24, 22)

In [ ]:
list(shared_features)[:20]

['884_628176',
 '884_n9216.000',
 '813_n6.600',
 '764_1285872',
 '283_963847',
 'categoryid_1338',
 '6_9705',
 '112_679677',
 '159_519769',
 '698_9705',
 '839_963847',
 '227_9705',
 '139_1161106',
 '884_n12288.000',
 '761_190776',
 '761_n24.000',
 'available_0',
 '678_963847',
 '139_60404',
 '813_404925']

In [ ]:
def recommend_similar_items(item_id, k=10):
    target_index = item_to_index.get(int(item_id))

    if target_index is None:
        return pd.DataFrame(
            columns=["itemid", "similarity"]
        )

    similarities = (
        normalized_item_matrix[target_index]
        @ normalized_item_matrix.T
    ).toarray().ravel()

    # Do not recommend the source item itself
    similarities[target_index] = -1

    # We only need the top k, not a complete sort of 417k values
    if k < len(similarities):
        top_indices = np.argpartition(
            similarities,
            -k
        )[-k:]

        top_indices = top_indices[
            np.argsort(similarities[top_indices])[::-1]
        ]
    else:
        top_indices = np.argsort(similarities)[::-1]

    return pd.DataFrame({
        "itemid": item_ids_full[top_indices],
        "similarity": similarities[top_indices]
    }).reset_index(drop=True)

In [ ]:
recommend_similar_items(460429, k=5)

,itemid,similarity
0,100656,0.916667
1,440711,0.893819
2,238891,0.875000
3,367270,0.875000
4,402554,0.875000


In [ ]:
interaction_weights = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5
}

In [ ]:
train_cf = train_data[
    ["visitorid", "itemid", "event"]
].copy()

train_cf["weight"] = train_cf["event"].map(interaction_weights)

In [ ]:
train_cf.head(10)

,visitorid,itemid,event,weight
0,257597,355908,view,1
1,992329,248676,view,1
2,111016,318965,view,1
3,483717,253185,view,1
4,951259,367447,view,1
5,972639,22556,view,1
6,810725,443030,view,1
7,794181,439202,view,1
8,824915,428805,view,1
9,339335,82389,view,1


In [ ]:
user_item_strength = (
    train_cf
    .groupby(["visitorid", "itemid"])["weight"]
    .sum()
    .reset_index()
)

NameError: name 'train_cf' is not defined

In [ ]:
user_item_strength.head(10)

,visitorid,itemid,weight
0,1,72028,1
1,2,216305,2
2,2,259884,1
3,2,325215,3
4,2,342816,2
5,3,385090,1
6,5,61396,1
7,6,65273,3
8,6,253615,1
9,6,344723,4


In [ ]:
user_item_strength.shape

(1905695, 3)

In [ ]:
from scipy.sparse import csr_matrix

In [ ]:
user_codes, user_ids = pd.factorize(user_item_strength["visitorid"])
item_codes_cf, item_ids_cf = pd.factorize(user_item_strength["itemid"])

In [ ]:
interaction_matrix = csr_matrix(
    (
        user_item_strength["weight"].astype(float),
        (user_codes, item_codes_cf)
    ),
    shape=(len(user_ids), len(item_ids_cf))
)

In [ ]:
interaction_matrix.shape

(1250856, 222982)

In [ ]:
interaction_matrix.nnz

1905695

In [ ]:
item_user_matrix = interaction_matrix.T

In [ ]:
item_user_matrix.shape

(222982, 1250856)

In [ ]:
target_item_cf = 460429

target_index_cf = np.where(item_ids_cf == target_item_cf)[0][0]
target_index_cf

np.int64(209842)

In [ ]:
cf_similarities = cosine_similarity(
    item_user_matrix[target_index_cf],
    item_user_matrix
).flatten()

In [ ]:
cf_similarities.shape

(222982,)

In [ ]:
top_indices_cf = cf_similarities.argsort()[::-1]

top_indices_cf = top_indices_cf[
    top_indices_cf != target_index_cf
]

top_10_indices_cf = top_indices_cf[:10]

top_10_items_cf = item_ids_cf[top_10_indices_cf]
top_10_scores_cf = cf_similarities[top_10_indices_cf]

top_10_cf = pd.DataFrame({
    "itemid": top_10_items_cf,
    "cf_similarity": top_10_scores_cf
})

top_10_cf

,itemid,cf_similarity
0,235724,0.0
1,288026,0.0
2,226080,0.0
3,392987,0.0
4,11803,0.0
5,404453,0.0
6,384891,0.0
7,356810,0.0
8,323085,0.0
9,211513,0.0


In [ ]:
target_item_vector = item_user_matrix[target_index_cf]

target_item_vector.nnz

1

In [ ]:
target_user_indices = target_item_vector.nonzero()[1]

target_user_ids = user_ids[target_user_indices]

target_user_ids

Index([1185727], dtype='int64')

In [ ]:
user_history_counts = (
    user_item_strength[
        user_item_strength["visitorid"].isin(target_user_ids)
    ]
    .groupby("visitorid")["itemid"]
    .nunique()
)

user_history_counts

visitorid
1185727    1
Name: itemid, dtype: int64

In [ ]:
item_user_counts = (
    user_item_strength
    .groupby("itemid")["visitorid"]
    .nunique()
    .sort_values(ascending=False)
)

item_user_counts.head(10)

itemid
187946    2284
5411      1936
370653    1470
298009    1251
96924     1183
219512    1168
461686    1149
335975    1089
309778    1050
151444     989
Name: visitorid, dtype: int64

In [ ]:
target_item_cf = 187946

target_index_cf = np.where(
    item_ids_cf == target_item_cf
)[0][0]

target_index_cf

np.int64(389)

In [ ]:
cf_similarities = cosine_similarity(
    item_user_matrix[target_index_cf],
    item_user_matrix
).flatten()

In [ ]:
top_indices_cf = cf_similarities.argsort()[::-1]

top_indices_cf = top_indices_cf[
    top_indices_cf != target_index_cf
]

top_10_indices_cf = top_indices_cf[:10]

top_10_cf = pd.DataFrame({
    "itemid": item_ids_cf[top_10_indices_cf],
    "cf_similarity": cf_similarities[top_10_indices_cf]
})

top_10_cf

,itemid,cf_similarity
0,311292,0.074688
1,317786,0.073922
2,46503,0.060357
3,22913,0.050102
4,284176,0.034426
5,193674,0.032009
6,157105,0.028453
7,127784,0.023730
8,76331,0.022121
9,350262,0.021958


In [ ]:
target_users = set(
    user_item_strength.loc[
        user_item_strength["itemid"] == 187946,
        "visitorid"
    ]
)

match_users = set(
    user_item_strength.loc[
        user_item_strength["itemid"] == 311292,
        "visitorid"
    ]
)

shared_users = target_users & match_users

len(target_users), len(match_users), len(shared_users)

(2284, 21, 1)

In [ ]:
def recommend_cf_items(item_id, k=10):
    matches = np.where(item_ids_cf == item_id)[0]

    if len(matches) == 0:
        return pd.DataFrame(
            columns=["itemid", "cf_similarity"]
        )

    target_index = matches[0]

    similarities = cosine_similarity(
        item_user_matrix[target_index],
        item_user_matrix
    ).flatten()

    sorted_indices = similarities.argsort()[::-1]

    sorted_indices = sorted_indices[
        sorted_indices != target_index
    ]

    # Remove items with zero collaborative similarity
    sorted_indices = sorted_indices[
        similarities[sorted_indices] > 0
    ]

    top_indices = sorted_indices[:k]

    return pd.DataFrame({
        "itemid": item_ids_cf[top_indices],
        "cf_similarity": similarities[top_indices]
    }).reset_index(drop=True)

In [ ]:
recommend_cf_items(187946, k=10)

,itemid,cf_similarity
0,311292,0.074688
1,317786,0.073922
2,46503,0.060357
3,22913,0.050102
4,284176,0.034426
5,193674,0.032009
6,157105,0.028453
7,127784,0.023730
8,76331,0.022121
9,350262,0.021958


In [ ]:
user_history_size = (
    user_item_strength
    .groupby("visitorid")["itemid"]
    .nunique()
    .sort_values(ascending=False)
)

user_history_size.head(10)

visitorid
1150086    3334
530559     1979
892013     1738
895999     1446
371606     1399
163561     1314
286616     1230
684514     1187
1297062    1146
861299     1100
Name: itemid, dtype: int64

In [ ]:
example_user = user_history_size.index[0]
example_user

np.int64(1150086)

In [ ]:
example_user_history = (
    user_item_strength[
        user_item_strength["visitorid"] == example_user
    ]
    .sort_values("weight", ascending=False)
)

example_user_history.head(20)

,visitorid,itemid,weight
1560237,1150086,119736,41
1561503,1150086,294676,38
1562671,1150086,461686,36
1560008,1150086,89323,34
1561169,1150086,248455,32
1560576,1150086,166306,31
1561625,1150086,310944,28
1562154,1150086,384302,27
1561429,1150086,283923,25
1560447,1150086,148103,25


In [ ]:
top_user_items = (
    example_user_history
    .head(20)
)

top_user_items

,visitorid,itemid,weight
1560237,1150086,119736,41
1561503,1150086,294676,38
1562671,1150086,461686,36
1560008,1150086,89323,34
1561169,1150086,248455,32
1560576,1150086,166306,31
1561625,1150086,310944,28
1562154,1150086,384302,27
1561429,1150086,283923,25
1560447,1150086,148103,25


In [ ]:
from collections import defaultdict

candidate_scores = defaultdict(float)

for _, row in top_user_items.iterrows():
    source_item = row["itemid"]
    source_weight = row["weight"]

    cf_neighbors = recommend_cf_items(source_item, k=20)

    for _, neighbor in cf_neighbors.iterrows():
        candidate_item = neighbor["itemid"]
        similarity = neighbor["cf_similarity"]

        candidate_scores[candidate_item] += (
            source_weight * similarity
        )

In [ ]:
seen_items = set(
    example_user_history["itemid"]
)

for seen_item in seen_items:
    candidate_scores.pop(seen_item, None)

In [ ]:
personalized_cf = pd.DataFrame(
    candidate_scores.items(),
    columns=["itemid", "personalized_cf_score"]
)

personalized_cf = (
    personalized_cf
    .sort_values(
        "personalized_cf_score",
        ascending=False
    )
    .reset_index(drop=True)
)

personalized_cf.head(10)

,itemid,personalized_cf_score
0,425521.0,23.183291
1,75510.0,22.268644
2,442725.0,21.178285
3,92186.0,20.716148
4,190806.0,20.540181
5,171878.0,12.150019
6,136302.0,11.550275
7,6022.0,11.550275
8,126303.0,11.310169
9,301327.0,10.209098


In [ ]:
personalized_cf["itemid"] = personalized_cf["itemid"].astype(int)

In [ ]:
personalized_cf.head(10)

,itemid,personalized_cf_score
0,425521,23.183291
1,75510,22.268644
2,442725,21.178285
3,92186,20.716148
4,190806,20.540181
5,171878,12.150019
6,136302,11.550275
7,6022,11.550275
8,126303,11.310169
9,301327,10.209098


In [ ]:
def recommend_for_user_cf(
    user_id,
    top_history_items=20,
    neighbors_per_item=20,
    k=10
):
    user_history = user_item_strength[
        user_item_strength["visitorid"] == user_id
    ].sort_values(
        "weight",
        ascending=False
    )

    if user_history.empty:
        return pd.DataFrame(
            columns=["itemid", "personalized_cf_score"]
        )

    strongest_history = user_history.head(top_history_items)

    candidate_scores = defaultdict(float)

    for _, row in strongest_history.iterrows():
        source_item = int(row["itemid"])
        source_weight = row["weight"]

        neighbors = recommend_cf_items(
            source_item,
            k=neighbors_per_item
        )

        for _, neighbor in neighbors.iterrows():
            candidate_item = int(neighbor["itemid"])
            similarity = neighbor["cf_similarity"]

            candidate_scores[candidate_item] += (
                source_weight * similarity
            )

    seen_items = set(
        user_history["itemid"].astype(int)
    )

    for seen_item in seen_items:
        candidate_scores.pop(seen_item, None)

    if not candidate_scores:
        return pd.DataFrame(
            columns=["itemid", "personalized_cf_score"]
        )

    recommendations = pd.DataFrame(
        candidate_scores.items(),
        columns=["itemid", "personalized_cf_score"]
    )

    recommendations["itemid"] = (
        recommendations["itemid"].astype(int)
    )

    return (
        recommendations
        .sort_values(
            "personalized_cf_score",
            ascending=False
        )
        .head(k)
        .reset_index(drop=True)
    )

In [ ]:
recommend_for_user_cf(
    1150086,
    top_history_items=20,
    neighbors_per_item=20,
    k=10
)

,itemid,personalized_cf_score
0,425521,23.183291
1,75510,22.268644
2,442725,21.178285
3,92186,20.716148
4,190806,20.540181
5,171878,12.150019
6,136302,11.550275
7,6022,11.550275
8,126303,11.310169
9,301327,10.209098


In [ ]:
example_eval_user = 1150086

In [ ]:
user_recs = recommend_for_user_cf(
    example_eval_user,
    top_history_items=20,
    neighbors_per_item=20,
    k=10
)

user_recs

,itemid,personalized_cf_score
0,425521,23.183291
1,75510,22.268644
2,442725,21.178285
3,92186,20.716148
4,190806,20.540181
5,171878,12.150019
6,136302,11.550275
7,6022,11.550275
8,126303,11.310169
9,301327,10.209098


In [ ]:
actual_items = set(
    test_data.loc[
        test_data["visitorid"] == example_eval_user,
        "itemid"
    ]
)

actual_items

{909,
 1538,
 1590,
 2711,
 3099,
 3902,
 4042,
 4482,
 11310,
 13302,
 13747,
 17698,
 19934,
 20582,
 23325,
 24246,
 24417,
 24509,
 24728,
 25027,
 25383,
 25590,
 25680,
 25762,
 27020,
 27766,
 27875,
 28016,
 28054,
 28867,
 31058,
 31605,
 31900,
 31952,
 32287,
 34560,
 34949,
 36184,
 36368,
 36464,
 36535,
 37011,
 37658,
 40359,
 41673,
 42210,
 43215,
 44045,
 44986,
 45882,
 46130,
 46232,
 46300,
 46938,
 47138,
 47586,
 48867,
 48921,
 49466,
 49692,
 50901,
 51650,
 52635,
 52832,
 53005,
 54046,
 54982,
 56100,
 56323,
 56630,
 57060,
 58160,
 58627,
 58676,
 58775,
 60138,
 61203,
 61893,
 62421,
 62502,
 63294,
 63757,
 65273,
 65703,
 66978,
 67481,
 67811,
 68128,
 68814,
 69427,
 70662,
 70855,
 71836,
 72944,
 73056,
 73617,
 73643,
 74438,
 75138,
 75141,
 75552,
 75764,
 76319,
 77143,
 78172,
 79056,
 80173,
 80858,
 81453,
 81998,
 82586,
 83314,
 83556,
 85415,
 85914,
 86348,
 87077,
 87090,
 87187,
 88432,
 89323,
 89775,
 91206,
 91888,
 92765,
 95811,
 

In [ ]:
recommended_items = set(
    user_recs["itemid"]
)

hits = recommended_items & actual_items

hits

set()

In [ ]:
len(recommended_items), len(actual_items), len(hits)

(10, 668, 0)

In [ ]:
def evaluate_cf_user(user_id, k=10):
    recommendations = recommend_for_user_cf(
        user_id,
        top_history_items=20,
        neighbors_per_item=20,
        k=k
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
evaluate_cf_user(1150086)

{'visitorid': 1150086,
 'recommendations': 10,
 'actual_items': 668,
 'hits': 0,
 'precision@10': 0.0,
 'recall@10': 0.0}

In [ ]:
sample_users = eligible_users[:25]

sample_results = []

for user_id in sample_users:
    result = evaluate_cf_user(user_id)
    sample_results.append(result)

sample_results_df = pd.DataFrame(sample_results)

sample_results_df

,visitorid,recommendations,actual_items,hits,precision@10,recall@10
0,155,10,8,0,0.0,0.0
1,484,10,4,0,0.0,0.0
2,1764,10,1,0,0.0,0.0
3,2004,10,2,0,0.0,0.0
4,2043,10,1,0,0.0,0.0
5,2194,10,20,0,0.0,0.0
6,2432,10,1,0,0.0,0.0
7,2783,10,2,0,0.0,0.0
8,2857,10,1,0,0.0,0.0
9,2866,2,1,0,0.0,0.0


In [ ]:
sample_results_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.0
recall@10       0.0
hits            0.0
dtype: float64

In [ ]:
(sample_results_df["hits"] > 0).sum()

np.int64(0)

In [ ]:
test_items = set(test_data["itemid"].astype(int))

sample_results_debug = []

for user_id in sample_users:
    recs = recommend_for_user_cf(
        user_id,
        top_history_items=20,
        neighbors_per_item=20,
        k=10
    )

    rec_items = set(recs["itemid"].astype(int))

    sample_results_debug.append({
        "visitorid": user_id,
        "num_recs": len(rec_items),
        "recs_present_in_test_period": len(rec_items & test_items)
    })

pd.DataFrame(sample_results_debug)

,visitorid,num_recs,recs_present_in_test_period
0,155,10,4
1,484,10,6
2,1764,10,4
3,2004,10,3
4,2043,10,6
5,2194,10,6
6,2432,10,2
7,2783,10,1
8,2857,10,5
9,2866,2,0


In [ ]:
debug_df = pd.DataFrame(sample_results_debug)

debug_df["recs_present_in_test_period"].mean()

np.float64(3.0)

In [ ]:
(debug_df["recs_present_in_test_period"] == 0).sum()

np.int64(4)

In [ ]:
recent_start = pd.Timestamp("2015-08-02")

recent_train_items = set(
    train_data.loc[
        train_data["datetime"] >= recent_start,
        "itemid"
    ].astype(int)
)

len(recent_train_items)

NameError: name 'pd' is not defined

In [ ]:
recent_debug = []

for user_id in sample_users:
    recs = recommend_for_user_cf(
        user_id,
        top_history_items=20,
        neighbors_per_item=20,
        k=10
    )

    rec_items = set(recs["itemid"].astype(int))

    recent_debug.append({
        "visitorid": user_id,
        "num_recs": len(rec_items),
        "recent_active_recs": len(rec_items & recent_train_items)
    })

recent_debug_df = pd.DataFrame(recent_debug)

recent_debug_df

,visitorid,num_recs,recent_active_recs
0,155,10,7
1,484,10,3
2,1764,10,5
3,2004,10,9
4,2043,10,4
5,2194,10,7
6,2432,10,0
7,2783,10,0
8,2857,10,7
9,2866,2,2


In [ ]:
recent_debug_df["recent_active_recs"].mean()

np.float64(4.8)

In [ ]:
def recommend_for_user_cf_recent(
    user_id,
    top_history_items=20,
    neighbors_per_item=20,
    k=10
):
    user_history = user_item_strength[
        user_item_strength["visitorid"] == user_id
    ].sort_values(
        "weight",
        ascending=False
    )

    if user_history.empty:
        return pd.DataFrame(
            columns=["itemid", "personalized_cf_score"]
        )

    strongest_history = user_history.head(top_history_items)

    candidate_scores = defaultdict(float)

    for _, row in strongest_history.iterrows():
        source_item = int(row["itemid"])
        source_weight = row["weight"]

        neighbors = recommend_cf_items(
            source_item,
            k=neighbors_per_item
        )

        for _, neighbor in neighbors.iterrows():
            candidate_item = int(neighbor["itemid"])
            similarity = neighbor["cf_similarity"]

            candidate_scores[candidate_item] += (
                source_weight * similarity
            )

    seen_items = set(
        user_history["itemid"].astype(int)
    )

    for seen_item in seen_items:
        candidate_scores.pop(seen_item, None)

    # New step:
    # keep only items that were active recently
    candidate_scores = {
        item_id: score
        for item_id, score in candidate_scores.items()
        if item_id in recent_train_items
    }

    if not candidate_scores:
        return pd.DataFrame(
            columns=["itemid", "personalized_cf_score"]
        )

    recommendations = pd.DataFrame(
        candidate_scores.items(),
        columns=["itemid", "personalized_cf_score"]
    )

    recommendations["itemid"] = (
        recommendations["itemid"].astype(int)
    )

    return (
        recommendations
        .sort_values(
            "personalized_cf_score",
            ascending=False
        )
        .head(k)
        .reset_index(drop=True)
    )

In [ ]:
recommend_for_user_cf_recent(
    1150086,
    top_history_items=20,
    neighbors_per_item=20,
    k=10
)

,itemid,personalized_cf_score
0,425521,23.183291
1,75510,22.268644
2,442725,21.178285
3,190806,20.540181
4,171878,12.150019
5,126303,11.310169
6,301327,10.209098
7,183756,9.998642
8,455427,8.962934
9,40630,8.843986


In [ ]:
def evaluate_cf_user_recent(user_id, k=10):
    recommendations = recommend_for_user_cf_recent(
        user_id,
        top_history_items=20,
        neighbors_per_item=20,
        k=k
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
recent_results = []

for user_id in sample_users:
    result = evaluate_cf_user_recent(user_id)
    recent_results.append(result)

recent_results_df = pd.DataFrame(recent_results)

recent_results_df

,visitorid,recommendations,actual_items,hits,precision@10,recall@10
0,155,10,8,0,0.0,0.0
1,484,10,4,0,0.0,0.0
2,1764,10,1,0,0.0,0.0
3,2004,10,2,0,0.0,0.0
4,2043,10,1,0,0.0,0.0
5,2194,10,20,0,0.0,0.0
6,2432,10,1,0,0.0,0.0
7,2783,10,2,0,0.0,0.0
8,2857,10,1,0,0.0,0.0
9,2866,2,1,0,0.0,0.0


In [ ]:
recent_results_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.0
recall@10       0.0
hits            0.0
dtype: float64

In [ ]:
(recent_results_df["hits"] > 0).sum()

np.int64(0)

In [ ]:
def recommend_for_user_cf_wide(
    user_id,
    top_history_items=50,
    neighbors_per_item=50,
    k=10
):
    user_history = user_item_strength[
        user_item_strength["visitorid"] == user_id
    ].sort_values(
        "weight",
        ascending=False
    )

    if user_history.empty:
        return pd.DataFrame(
            columns=["itemid", "personalized_cf_score"]
        )

    strongest_history = user_history.head(top_history_items)

    candidate_scores = defaultdict(float)

    for _, row in strongest_history.iterrows():
        source_item = int(row["itemid"])
        source_weight = row["weight"]

        neighbors = recommend_cf_items(
            source_item,
            k=neighbors_per_item
        )

        for _, neighbor in neighbors.iterrows():
            candidate_item = int(neighbor["itemid"])
            similarity = neighbor["cf_similarity"]

            candidate_scores[candidate_item] += (
                source_weight * similarity
            )

    seen_items = set(
        user_history["itemid"].astype(int)
    )

    for seen_item in seen_items:
        candidate_scores.pop(seen_item, None)

    if not candidate_scores:
        return pd.DataFrame(
            columns=["itemid", "personalized_cf_score"]
        )

    recommendations = pd.DataFrame(
        candidate_scores.items(),
        columns=["itemid", "personalized_cf_score"]
    )

    recommendations["itemid"] = recommendations["itemid"].astype(int)

    return (
        recommendations
        .sort_values(
            "personalized_cf_score",
            ascending=False
        )
        .head(k)
        .reset_index(drop=True)
    )

In [ ]:
def evaluate_cf_user_wide(user_id, k=10):
    recommendations = recommend_for_user_cf_wide(
        user_id,
        top_history_items=50,
        neighbors_per_item=50,
        k=k
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
wide_results = []

for user_id in sample_users:
    wide_results.append(
        evaluate_cf_user_wide(user_id)
    )

wide_results_df = pd.DataFrame(wide_results)

wide_results_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.0
recall@10       0.0
hits            0.0
dtype: float64

In [ ]:
(wide_results_df["hits"] > 0).sum()

np.int64(0)

In [ ]:
def get_cf_candidate_pool(
    user_id,
    top_history_items=50,
    neighbors_per_item=50
):
    user_history = user_item_strength[
        user_item_strength["visitorid"] == user_id
    ].sort_values(
        "weight",
        ascending=False
    )

    if user_history.empty:
        return set()

    strongest_history = user_history.head(top_history_items)

    candidate_items = set()

    for _, row in strongest_history.iterrows():
        source_item = int(row["itemid"])

        neighbors = recommend_cf_items(
            source_item,
            k=neighbors_per_item
        )

        candidate_items.update(
            neighbors["itemid"].astype(int)
        )

    seen_items = set(
        user_history["itemid"].astype(int)
    )

    candidate_items = candidate_items - seen_items

    return candidate_items

In [ ]:
candidate_pool = get_cf_candidate_pool(1150086)

len(candidate_pool)

195

In [ ]:
actual_items = set(
    test_data.loc[
        test_data["visitorid"] == 1150086,
        "itemid"
    ].astype(int)
)

candidate_hits = candidate_pool & actual_items

len(candidate_hits), len(candidate_pool), len(actual_items)

(3, 195, 668)

In [ ]:
candidate_diagnostics = []

for user_id in sample_users:
    candidate_pool = get_cf_candidate_pool(user_id)

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    candidate_hits = candidate_pool & actual_items

    candidate_diagnostics.append({
        "visitorid": user_id,
        "candidate_pool_size": len(candidate_pool),
        "actual_items": len(actual_items),
        "candidate_hits": len(candidate_hits)
    })

candidate_diagnostics_df = pd.DataFrame(candidate_diagnostics)

candidate_diagnostics_df

,visitorid,candidate_pool_size,actual_items,candidate_hits
0,155,231,8,0
1,484,184,4,0
2,1764,279,1,0
3,2004,303,2,0
4,2043,20,1,0
5,2194,984,20,1
6,2432,104,1,0
7,2783,280,2,0
8,2857,50,1,0
9,2866,2,1,0


In [ ]:
candidate_diagnostics_df[
    ["candidate_pool_size", "candidate_hits"]
].mean()

candidate_pool_size    186.92
candidate_hits           0.04
dtype: float64

In [ ]:
(candidate_diagnostics_df["candidate_hits"] > 0).sum()

np.int64(1)

In [ ]:
cf_known_items = set(
    pd.Series(item_ids_cf).astype(int)
)

len(cf_known_items)

222982

In [ ]:
recommendability_debug = []

for user_id in sample_users:

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    known_actual_items = actual_items & cf_known_items

    recommendability_debug.append({
        "visitorid": user_id,
        "actual_items": len(actual_items),
        "actual_items_known_to_cf": len(known_actual_items),
        "unknown_new_items": len(actual_items - cf_known_items)
    })

recommendability_df = pd.DataFrame(recommendability_debug)

recommendability_df

,visitorid,actual_items,actual_items_known_to_cf,unknown_new_items
0,155,8,7,1
1,484,4,4,0
2,1764,1,1,0
3,2004,2,2,0
4,2043,1,1,0
5,2194,20,20,0
6,2432,1,1,0
7,2783,2,2,0
8,2857,1,1,0
9,2866,1,0,1


In [ ]:
recommendability_df[
    ["actual_items", "actual_items_known_to_cf", "unknown_new_items"]
].mean()

actual_items                2.40
actual_items_known_to_cf    2.32
unknown_new_items           0.08
dtype: float64

In [ ]:
(
    recommendability_df["actual_items_known_to_cf"] == 0
).sum()

np.int64(1)

In [ ]:
user_item_recent = (
    train_data
    .groupby(["visitorid", "itemid"])
    .agg(
        weight=("event", lambda x: x.map(interaction_weights).sum()),
        last_interaction=("datetime", "max")
    )
    .reset_index()
)

user_item_recent.head()

,visitorid,itemid,weight,last_interaction
0,1,72028,1,2015-08-13 17:46:06.444
1,2,216305,2,2015-08-07 18:17:43.170
2,2,259884,1,2015-08-07 17:56:52.664
3,2,325215,3,2015-08-07 18:20:57.845
4,2,342816,2,2015-08-07 18:17:24.375


In [ ]:
def recommend_for_user_cf_recent_history(
    user_id,
    recent_history_items=50,
    neighbors_per_item=50,
    k=10
):
    user_history = user_item_recent[
        user_item_recent["visitorid"] == user_id
    ].sort_values(
        "last_interaction",
        ascending=False
    )

    if user_history.empty:
        return pd.DataFrame(
            columns=["itemid", "personalized_cf_score"]
        )

    recent_history = user_history.head(recent_history_items)

    candidate_scores = defaultdict(float)

    for _, row in recent_history.iterrows():
        source_item = int(row["itemid"])
        source_weight = row["weight"]

        neighbors = recommend_cf_items(
            source_item,
            k=neighbors_per_item
        )

        for _, neighbor in neighbors.iterrows():
            candidate_item = int(neighbor["itemid"])
            similarity = neighbor["cf_similarity"]

            candidate_scores[candidate_item] += (
                source_weight * similarity
            )

    seen_items = set(
        user_history["itemid"].astype(int)
    )

    for seen_item in seen_items:
        candidate_scores.pop(seen_item, None)

    if not candidate_scores:
        return pd.DataFrame(
            columns=["itemid", "personalized_cf_score"]
        )

    recommendations = pd.DataFrame(
        candidate_scores.items(),
        columns=["itemid", "personalized_cf_score"]
    )

    recommendations["itemid"] = recommendations["itemid"].astype(int)

    return (
        recommendations
        .sort_values(
            "personalized_cf_score",
            ascending=False
        )
        .head(k)
        .reset_index(drop=True)
    )

In [ ]:
def evaluate_cf_user_recent_history(user_id, k=10):
    recommendations = recommend_for_user_cf_recent_history(
        user_id,
        recent_history_items=50,
        neighbors_per_item=50,
        k=k
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
recent_history_results = []

for user_id in sample_users:
    recent_history_results.append(
        evaluate_cf_user_recent_history(user_id)
    )

recent_history_results_df = pd.DataFrame(
    recent_history_results
)

recent_history_results_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.0
recall@10       0.0
hits            0.0
dtype: float64

In [ ]:
(
    recent_history_results_df["hits"] > 0
).sum()

np.int64(0)

In [ ]:
def recommend_for_user_content(
    user_id,
    top_history_items=20,
    neighbors_per_item=20,
    k=10
):
    user_history = user_item_strength[
        user_item_strength["visitorid"] == user_id
    ].sort_values(
        "weight",
        ascending=False
    )

    if user_history.empty:
        return pd.DataFrame(
            columns=["itemid", "personalized_content_score"]
        )

    strongest_history = user_history.head(top_history_items)

    candidate_scores = defaultdict(float)

    for _, row in strongest_history.iterrows():
        source_item = int(row["itemid"])
        source_weight = row["weight"]

        neighbors = get_content_neighbors(
        source_item,
        k=neighbors_per_item
    )

        for _, neighbor in neighbors.iterrows():
            candidate_item = int(neighbor["itemid"])
            similarity = neighbor["similarity"]

            candidate_scores[candidate_item] += (
                source_weight * similarity
            )

    seen_items = set(
        user_history["itemid"].astype(int)
    )

    for seen_item in seen_items:
        candidate_scores.pop(seen_item, None)

    if not candidate_scores:
        return pd.DataFrame(
            columns=["itemid", "personalized_content_score"]
        )

    recommendations = pd.DataFrame(
        candidate_scores.items(),
        columns=["itemid", "personalized_content_score"]
    )

    recommendations["itemid"] = recommendations["itemid"].astype(int)

    return (
        recommendations
        .sort_values(
            "personalized_content_score",
            ascending=False
        )
        .head(k)
        .reset_index(drop=True)
    )

In [ ]:
def evaluate_content_user(user_id, k=10):
    recommendations = recommend_for_user_content(
        user_id,
        top_history_items=20,
        neighbors_per_item=20,
        k=k
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
content_results = []

for user_id in sample_users:
    content_results.append(
        evaluate_content_user(user_id)
    )

content_results_df = pd.DataFrame(content_results)

content_results_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.004
recall@10       0.040
hits            0.040
dtype: float64

In [ ]:
(content_results_df["hits"] > 0).sum()

np.int64(1)

In [ ]:
content_results_df[
    content_results_df["hits"] > 0
]

,visitorid,recommendations,actual_items,hits,precision@10,recall@10
24,8737,10,1,1,0.1,1.0


In [ ]:
sample_users_100 = eligible_users[:100]

content_results_100 = []

for user_id in sample_users_100:
    content_results_100.append(
        evaluate_content_user(user_id)
    )

content_results_100_df = pd.DataFrame(
    content_results_100
)

In [ ]:
content_results_100_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.002
recall@10       0.015
hits            0.020
dtype: float64

In [ ]:
(
    content_results_100_df["hits"] > 0
).sum()

np.int64(2)

In [ ]:
def recommend_for_user_content_recent_history(
    user_id,
    recent_history_items=20,
    neighbors_per_item=20,
    k=10
):
    user_history = user_item_recent[
        user_item_recent["visitorid"] == user_id
    ].sort_values(
        "last_interaction",
        ascending=False
    )

    if user_history.empty:
        return pd.DataFrame(
            columns=["itemid", "personalized_content_score"]
        )

    recent_history = user_history.head(recent_history_items)

    candidate_scores = defaultdict(float)

    for _, row in recent_history.iterrows():
        source_item = int(row["itemid"])
        source_weight = row["weight"]

        neighbors = recommend_similar_items(
            source_item,
            k=neighbors_per_item
        )

        for _, neighbor in neighbors.iterrows():
            candidate_item = int(neighbor["itemid"])
            similarity = neighbor["similarity"]

            candidate_scores[candidate_item] += (
                source_weight * similarity
            )

    seen_items = set(
        user_history["itemid"].astype(int)
    )

    for seen_item in seen_items:
        candidate_scores.pop(seen_item, None)

    if not candidate_scores:
        return pd.DataFrame(
            columns=["itemid", "personalized_content_score"]
        )

    recommendations = pd.DataFrame(
        candidate_scores.items(),
        columns=["itemid", "personalized_content_score"]
    )

    recommendations["itemid"] = recommendations["itemid"].astype(int)

    return (
        recommendations
        .sort_values(
            "personalized_content_score",
            ascending=False
        )
        .head(k)
        .reset_index(drop=True)
    )

In [ ]:
def evaluate_content_user_recent_history(user_id, k=10):
    recommendations = recommend_for_user_content_recent_history(
        user_id,
        recent_history_items=20,
        neighbors_per_item=20,
        k=k
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
sample_users_100 = eligible_users[:100]

In [ ]:
content_recent_results_100 = []

for user_id in sample_users_100:
    content_recent_results_100.append(
        evaluate_content_user_recent_history(user_id)
    )

content_recent_results_100_df = pd.DataFrame(
    content_recent_results_100
)

In [ ]:
sample_users_100 = eligible_users[:100]

In [ ]:
content_recent_results_100 = []

for user_id in sample_users_100:
    content_recent_results_100.append(
        evaluate_content_user_recent_history(user_id)
    )

content_recent_results_100_df = pd.DataFrame(
    content_recent_results_100
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
recommend_similar_items(460429, k=5)

,itemid,similarity
0,100656,0.916667
1,440711,0.893819
2,238891,0.875000
3,367270,0.875000
4,402554,0.875000


In [ ]:
filtered_pairs.shape

(22468165, 2)

In [ ]:
item_ids_full.shape

(417053,)

In [ ]:
full_item_feature_matrix.shape

(417053, 161379)

In [ ]:
recommend_similar_items(460429, k=5)

,itemid,similarity
0,100656,0.916667
1,440711,0.893819
2,238891,0.875000
3,367270,0.875000
4,402554,0.875000


In [ ]:
recommend_for_user_content_recent_history(
    8737,
    recent_history_items=20,
    neighbors_per_item=20,
    k=10
)

,itemid,personalized_content_score
0,29801,1.803279
1,230515,1.785185
2,148586,1.686008
3,73572,1.593549
4,86170,1.451802
5,217110,1.438513
6,433983,1.386902
7,96498,1.334006
8,201716,1.322753
9,194572,1.316087


In [ ]:
def evaluate_content_user_recent_history(user_id, k=10):
    recommendations = recommend_for_user_content_recent_history(
        user_id,
        recent_history_items=20,
        neighbors_per_item=20,
        k=k
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
sample_users_100 = eligible_users[:100]

content_recent_results_100 = []

for user_id in sample_users_100:
    content_recent_results_100.append(
        evaluate_content_user_recent_history(user_id)
    )

content_recent_results_100_df = pd.DataFrame(
    content_recent_results_100
)

In [ ]:
content_recent_results_100_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.002
recall@10       0.015
hits            0.020
dtype: float64

In [ ]:
(content_recent_results_100_df["hits"] > 0).sum()

np.int64(2)

In [ ]:
popular_items_top10 = (
    train_data[
        train_data["event"] == "transaction"
    ]["itemid"]
    .value_counts()
    .head(10)
    .index
    .astype(int)
    .tolist()
)

popular_items_top10

[461686, 119736, 213834, 445351, 48030, 7943, 420960, 17478, 248455, 312728]

In [ ]:
def evaluate_popularity_user(user_id, k=10):
    recommended_items = set(
        popular_items_top10[:k]
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
popularity_results_100 = []

for user_id in sample_users_100:
    popularity_results_100.append(
        evaluate_popularity_user(user_id)
    )

popularity_results_100_df = pd.DataFrame(
    popularity_results_100
)

In [ ]:
cf_results_100 = []

for user_id in sample_users_100:
    cf_results_100.append(
        evaluate_cf_user(user_id)
    )

cf_results_100_df = pd.DataFrame(
    cf_results_100
)

In [ ]:
def summarize_model(name, results_df):
    return {
        "model": name,
        "precision@10": results_df["precision@10"].mean(),
        "recall@10": results_df["recall@10"].mean(),
        "avg_hits": results_df["hits"].mean(),
        "users_with_hit": (results_df["hits"] > 0).sum(),
        "hit_rate": (results_df["hits"] > 0).mean()
    }

In [ ]:
model_comparison = pd.DataFrame([
    summarize_model(
        "Popularity",
        popularity_results_100_df
    ),
    
    summarize_model(
        "Collaborative Filtering",
        cf_results_100_df
    ),
    
    summarize_model(
        "Content - Strongest History",
        content_results_100_df
    ),
    
    summarize_model(
        "Content - Recent History",
        content_recent_results_100_df
    )
])

model_comparison

,model,precision@10,recall@10,avg_hits,users_with_hit,hit_rate
0,Popularity,0.001,0.003333,0.01,1,0.01
1,Collaborative Filtering,0.000,0.000000,0.00,0,0.00
2,Content - Strongest History,0.002,0.015000,0.02,2,0.02
3,Content - Recent History,0.002,0.015000,0.02,2,0.02


In [ ]:
purchase_popularity = (
    train_data[
        train_data["event"] == "transaction"
    ]["itemid"]
    .value_counts()
)

purchase_popularity.head()

itemid
461686    109
119736     91
213834     67
445351     43
48030      41
Name: count, dtype: int64

In [ ]:
def recommend_for_user_hybrid(
    user_id,
    top_history_items=20,
    content_neighbors=20,
    cf_neighbors=20,
    k=10,
    content_weight=0.65,
    popularity_weight=0.25,
    cf_weight=0.10
):
    user_history = user_item_strength[
        user_item_strength["visitorid"] == user_id
    ].sort_values(
        "weight",
        ascending=False
    )

    if user_history.empty:
        return pd.DataFrame(
            columns=[
                "itemid",
                "content_score",
                "cf_score",
                "popularity_score",
                "hybrid_score"
            ]
        )

    strongest_history = user_history.head(top_history_items)

    content_scores = defaultdict(float)
    cf_scores = defaultdict(float)

    # 1. Generate candidates from user history
    for _, row in strongest_history.iterrows():
        source_item = int(row["itemid"])
        source_weight = row["weight"]

        content_recs = recommend_similar_items(
            source_item,
            k=content_neighbors
        )

        for _, rec in content_recs.iterrows():
            candidate = int(rec["itemid"])
            content_scores[candidate] += (
                source_weight * rec["similarity"]
            )

        cf_recs = recommend_cf_items(
            source_item,
            k=cf_neighbors
        )

        for _, rec in cf_recs.iterrows():
            candidate = int(rec["itemid"])
            cf_scores[candidate] += (
                source_weight * rec["cf_similarity"]
            )

    # 2. Union of all candidates
    candidate_items = (
        set(content_scores.keys())
        | set(cf_scores.keys())
    )

    # add globally popular items too
    candidate_items |= set(
        purchase_popularity.head(50).index.astype(int)
    )

    # 3. Remove already-seen items
    seen_items = set(
        user_history["itemid"].astype(int)
    )

    candidate_items -= seen_items

    if not candidate_items:
        return pd.DataFrame()

    # 4. Build candidate table
    hybrid_df = pd.DataFrame({
        "itemid": list(candidate_items)
    })

    hybrid_df["content_score"] = hybrid_df["itemid"].map(
        content_scores
    ).fillna(0)

    hybrid_df["cf_score"] = hybrid_df["itemid"].map(
        cf_scores
    ).fillna(0)

    hybrid_df["popularity_score"] = hybrid_df["itemid"].map(
        purchase_popularity
    ).fillna(0)

    # 5. Normalize each score independently
    def minmax(series):
        if series.max() == series.min():
            return pd.Series(
                0.0,
                index=series.index
            )

        return (
            (series - series.min())
            /
            (series.max() - series.min())
        )

    hybrid_df["content_norm"] = minmax(
        hybrid_df["content_score"]
    )

    hybrid_df["cf_norm"] = minmax(
        hybrid_df["cf_score"]
    )

    hybrid_df["popularity_norm"] = minmax(
        hybrid_df["popularity_score"]
    )

    # 6. Weighted hybrid score
    hybrid_df["hybrid_score"] = (
        content_weight * hybrid_df["content_norm"]
        +
        popularity_weight * hybrid_df["popularity_norm"]
        +
        cf_weight * hybrid_df["cf_norm"]
    )

    return (
        hybrid_df
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(k)
        .reset_index(drop=True)
    )

In [ ]:
recommend_for_user_hybrid(
    8737,
    k=10
)

,itemid,content_score,cf_score,popularity_score,content_norm,cf_norm,popularity_norm,hybrid_score
0,230515,1.785185,0.102643,0.0,0.989966,0.366508,0.0,0.680129
1,29801,1.803279,0.000000,0.0,1.000000,0.000000,0.0,0.650000
2,148586,1.686008,0.000000,0.0,0.934968,0.000000,0.0,0.607729
3,73572,1.593549,0.000000,0.0,0.883695,0.000000,0.0,0.574402
4,137498,1.292341,0.161690,0.0,0.716662,0.577350,0.0,0.523565
5,86170,1.451802,0.000000,0.0,0.805090,0.000000,0.0,0.523309
6,217110,1.438513,0.000000,0.0,0.797721,0.000000,0.0,0.518519
7,433983,1.386902,0.000000,0.0,0.769100,0.000000,0.0,0.499915
8,96498,1.334006,0.000000,0.0,0.739767,0.000000,0.0,0.480848
9,201716,1.322753,0.000000,0.0,0.733527,0.000000,0.0,0.476792


In [ ]:
def evaluate_hybrid_user(user_id, k=10):
    recommendations = recommend_for_user_hybrid(
        user_id,
        k=k
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
hybrid_results_100 = []

for user_id in sample_users_100:
    hybrid_results_100.append(
        evaluate_hybrid_user(user_id)
    )

hybrid_results_100_df = pd.DataFrame(
    hybrid_results_100
)

In [ ]:
hybrid_results_100_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.002
recall@10       0.015
hits            0.020
dtype: float64

In [ ]:
(hybrid_results_100_df["hits"] > 0).sum()

np.int64(2)

In [ ]:
def evaluate_hybrid_user_weights(
    user_id,
    content_weight,
    popularity_weight,
    cf_weight,
    k=10
):
    recommendations = recommend_for_user_hybrid(
        user_id,
        k=k,
        content_weight=content_weight,
        popularity_weight=popularity_weight,
        cf_weight=cf_weight
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
weight_configs = {
    "A_80_15_05": (0.80, 0.15, 0.05),
    "B_70_20_10": (0.70, 0.20, 0.10),
    "C_60_30_10": (0.60, 0.30, 0.10),
    "D_90_10_00": (0.90, 0.10, 0.00)
}

In [ ]:
weight_results = []

for config_name, weights in weight_configs.items():
    content_w, popularity_w, cf_w = weights

    user_results = []

    for user_id in sample_users_100:
        user_results.append(
            evaluate_hybrid_user_weights(
                user_id,
                content_weight=content_w,
                popularity_weight=popularity_w,
                cf_weight=cf_w
            )
        )

    results_df = pd.DataFrame(user_results)

    weight_results.append({
        "config": config_name,
        "content_weight": content_w,
        "popularity_weight": popularity_w,
        "cf_weight": cf_w,
        "precision@10": results_df["precision@10"].mean(),
        "recall@10": results_df["recall@10"].mean(),
        "avg_hits": results_df["hits"].mean(),
        "users_with_hit": (results_df["hits"] > 0).sum(),
        "hit_rate": (results_df["hits"] > 0).mean()
    })

weight_comparison_df = pd.DataFrame(weight_results)

weight_comparison_df

,config,content_weight,popularity_weight,cf_weight,precision@10,recall@10,avg_hits,users_with_hit,hit_rate
0,A_80_15_05,0.8,0.15,0.05,0.002,0.015,0.02,2,0.02
1,B_70_20_10,0.7,0.20,0.10,0.002,0.015,0.02,2,0.02
2,C_60_30_10,0.6,0.30,0.10,0.002,0.015,0.02,2,0.02
3,D_90_10_00,0.9,0.10,0.00,0.002,0.015,0.02,2,0.02


In [ ]:
recent_purchase_popularity = (
    train_data[
        (train_data["datetime"] >= recent_start)
        &
        (train_data["event"] == "transaction")
    ]["itemid"]
    .value_counts()
)

recent_popular_top10 = (
    recent_purchase_popularity
    .head(10)
    .index
    .astype(int)
    .tolist()
)

recent_popular_top10

[461686, 119736, 248455, 320130, 334401, 546, 441852, 9877, 213834, 173653]

In [ ]:
def evaluate_recent_popularity_user(user_id, k=10):
    recommended_items = set(
        recent_popular_top10[:k]
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if recommended_items
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if actual_items
        else 0
    )

    return {
        "visitorid": user_id,
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
recent_pop_results_100 = []

for user_id in sample_users_100:
    recent_pop_results_100.append(
        evaluate_recent_popularity_user(user_id)
    )

recent_pop_results_100_df = pd.DataFrame(
    recent_pop_results_100
)

recent_pop_results_100_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.001
recall@10       0.001
hits            0.010
dtype: float64

In [ ]:
(recent_pop_results_100_df["hits"] > 0).sum()

np.int64(1)

In [ ]:
user_history_size = (
    user_item_strength
    .groupby("visitorid")["itemid"]
    .nunique()
)

In [ ]:
user_history_count = user_history_size.to_dict()

In [ ]:
new_users_in_train = 0

low_history_users = (
    (user_history_size >= 1)
    &
    (user_history_size <= 2)
).sum()

established_users = (
    user_history_size >= 3
).sum()

print("Low-history users:", low_history_users)
print("Established users:", established_users)

Low-history users: 1146757
Established users: 104099


In [ ]:
def recommend_for_low_history_user(user_id, k=10):
    user_history = user_item_strength[
        user_item_strength["visitorid"] == user_id
    ]

    if user_history.empty:
        return pd.DataFrame(
            columns=["itemid", "score", "source"]
        )

    seen_items = set(
        user_history["itemid"].astype(int)
    )

    candidate_scores = defaultdict(float)

    for _, row in user_history.iterrows():
        source_item = int(row["itemid"])
        source_weight = row["weight"]

        neighbors = get_content_neighbors(
        source_item,
        k=20
    )

        for _, rec in neighbors.iterrows():
            candidate_item = int(rec["itemid"])

            if candidate_item in seen_items:
                continue

            candidate_scores[candidate_item] += (
                source_weight * rec["similarity"]
            )

    content_recs = (
        pd.DataFrame(
            candidate_scores.items(),
            columns=["itemid", "score"]
        )
        .sort_values("score", ascending=False)
        if candidate_scores
        else pd.DataFrame(columns=["itemid", "score"])
    )

    content_recs["source"] = "content"

    selected = content_recs.head(k).copy()

    # Fill remaining slots using global popularity
    if len(selected) < k:
        already_selected = set(
            selected["itemid"].astype(int)
        )

        popular_fill = []

        for item_id in popular_items_top10:
            item_id = int(item_id)

            if (
                item_id not in seen_items
                and item_id not in already_selected
            ):
                popular_fill.append({
                    "itemid": item_id,
                    "score": 0.0,
                    "source": "popularity_fallback"
                })

            if len(selected) + len(popular_fill) >= k:
                break

        if popular_fill:
            selected = pd.concat(
                [
                    selected,
                    pd.DataFrame(popular_fill)
                ],
                ignore_index=True
            )

    return selected.head(k).reset_index(drop=True)

In [ ]:
low_history_example = user_history_size[
    (user_history_size >= 1)
    &
    (user_history_size <= 2)
].index[0]

low_history_example

np.int64(1)

In [ ]:
recommend_for_low_history_user(
    low_history_example,
    k=10
)

,itemid,score,source
0,461032,0.758947,content
1,276885,0.701493,content
2,323403,0.695701,content
3,247808,0.678233,content
4,42210,0.670979,content
5,169233,0.648745,content
6,192013,0.648745,content
7,26711,0.636396,content
8,15272,0.627572,content
9,50923,0.616441,content


In [ ]:
def recommend_for_user_final(user_id, k=10):
    history_count = user_history_count.get(user_id, 0)

    # CASE 1: Completely new user
    if history_count == 0:
        recommendations = []

        for item_id in weighted_popular_top10[:k]:
            recommendations.append({
                "itemid": int(item_id),
                "score": float(weighted_popularity.get(item_id, 0)),
                "source": "weighted_popularity_cold_start"
            })

        return pd.DataFrame(recommendations)

    # CASE 2: Low-history user
    if history_count <= 2:
        return recommend_for_low_history_user(
            user_id,
            k=k
        )

    # CASE 3: Established user
    recommendations = recommend_for_user_content(
        user_id,
        top_history_items=20,
        neighbors_per_item=20,
        k=k
    ).copy()

    if recommendations.empty:
        return recommend_for_low_history_user(
            user_id,
            k=k
        )

    recommendations = recommendations.rename(
        columns={
            "personalized_content_score": "score"
        }
    )

    recommendations["source"] = "personalized_content"

    return recommendations[
        ["itemid", "score", "source"]
    ]

In [ ]:
recommend_for_user_final(
    999999999,
    k=10
)

,itemid,score,source
0,461686,3350.0,weighted_popularity_cold_start
1,187946,2700.0,weighted_popularity_cold_start
2,5411,2192.0,weighted_popularity_cold_start
3,257040,1928.0,weighted_popularity_cold_start
4,309778,1779.0,weighted_popularity_cold_start
5,370653,1735.0,weighted_popularity_cold_start
6,7943,1674.0,weighted_popularity_cold_start
7,219512,1663.0,weighted_popularity_cold_start
8,298009,1538.0,weighted_popularity_cold_start
9,312728,1503.0,weighted_popularity_cold_start


In [ ]:
recommend_for_user_final(
    low_history_example,
    k=10
)

,itemid,score,source
0,461032,0.758947,content
1,276885,0.701493,content
2,323403,0.695701,content
3,247808,0.678233,content
4,42210,0.670979,content
5,169233,0.648745,content
6,192013,0.648745,content
7,26711,0.636396,content
8,15272,0.627572,content
9,50923,0.616441,content


In [ ]:
established_example = user_history_size[
    user_history_size >= 3
].index[0]

established_example

np.int64(2)

In [ ]:
recommend_for_user_final(
    established_example,
    k=10
)

,itemid,score,source
0,250233,4.138436,personalized_content
1,140853,4.070787,personalized_content
2,29940,4.000284,personalized_content
3,140706,3.983011,personalized_content
4,172103,3.897852,personalized_content
5,136560,3.708596,personalized_content
6,30573,3.612445,personalized_content
7,189119,3.605222,personalized_content
8,441734,3.571668,personalized_content
9,48072,3.568581,personalized_content


In [ ]:
recommend_for_user_final(
    low_history_example,
    k=10
)

,itemid,score,source
0,461032,0.758947,content
1,276885,0.701493,content
2,323403,0.695701,content
3,247808,0.678233,content
4,42210,0.670979,content
5,169233,0.648745,content
6,192013,0.648745,content
7,26711,0.636396,content
8,15272,0.627572,content
9,50923,0.616441,content


In [ ]:
test_user_ids = test_data["visitorid"].unique()

routing_segments = []

for user_id in test_user_ids:
    history_count = user_history_count.get(user_id, 0)

    if history_count == 0:
        segment = "new_user"
    elif history_count <= 2:
        segment = "low_history"
    else:
        segment = "established"

    routing_segments.append({
        "visitorid": user_id,
        "history_count": history_count,
        "segment": segment
    })

routing_segments_df = pd.DataFrame(routing_segments)

routing_segments_df["segment"].value_counts()

segment
new_user       156724
low_history     12523
established      4481
Name: count, dtype: int64

In [ ]:
def evaluate_final_user(user_id, k=10):
    recommendations = recommend_for_user_final(
        user_id,
        k=k
    )

    recommended_items = set(
        recommendations["itemid"].astype(int)
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if len(recommended_items) > 0
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if len(actual_items) > 0
        else 0
    )

    return {
        "visitorid": user_id,
        "recommendations": len(recommended_items),
        "actual_items": len(actual_items),
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
segment_samples = {}

for segment in ["new_user", "low_history", "established"]:
    segment_samples[segment] = (
        routing_segments_df[
            routing_segments_df["segment"] == segment
        ]["visitorid"]
        .head(100)
        .tolist()
    )

In [ ]:
segment_results = []

for segment, users in segment_samples.items():

    user_results = []

    for user_id in users:
        user_results.append(
            evaluate_final_user(user_id)
        )

    results_df = pd.DataFrame(user_results)

    segment_results.append({
        "segment": segment,
        "precision@10": results_df["precision@10"].mean(),
        "recall@10": results_df["recall@10"].mean(),
        "avg_hits": results_df["hits"].mean(),
        "users_with_hit": (results_df["hits"] > 0).sum(),
        "hit_rate": (results_df["hits"] > 0).mean()
    })

segment_comparison_df = pd.DataFrame(segment_results)

segment_comparison_df

,segment,precision@10,recall@10,avg_hits,users_with_hit,hit_rate
0,new_user,0.000,0.000000,0.00,0,0.00
1,low_history,0.011,0.053333,0.11,9,0.09
2,established,0.009,0.021092,0.09,7,0.07


In [ ]:
segment_samples_random = {}

for segment in ["new_user", "low_history", "established"]:

    segment_users = routing_segments_df[
        routing_segments_df["segment"] == segment
    ]

    segment_samples_random[segment] = (
        segment_users["visitorid"]
        .sample(
            n=100,
            random_state=42
        )
        .tolist()
    )

In [ ]:
random_segment_results = []

for segment, users in segment_samples_random.items():

    user_results = []

    for user_id in users:
        user_results.append(
            evaluate_final_user(user_id)
        )

    results_df = pd.DataFrame(user_results)

    random_segment_results.append({
        "segment": segment,
        "precision@10": results_df["precision@10"].mean(),
        "recall@10": results_df["recall@10"].mean(),
        "avg_hits": results_df["hits"].mean(),
        "users_with_hit": (results_df["hits"] > 0).sum(),
        "hit_rate": (results_df["hits"] > 0).mean()
    })

random_segment_comparison_df = pd.DataFrame(
    random_segment_results
)

random_segment_comparison_df

,segment,precision@10,recall@10,avg_hits,users_with_hit,hit_rate
0,new_user,0.001,0.005,0.01,1,0.01
1,low_history,0.005,0.030,0.05,4,0.04
2,established,0.002,0.007,0.02,2,0.02


In [ ]:
weighted_popularity = (
    train_data
    .assign(
        event_weight=train_data["event"].map(interaction_weights)
    )
    .groupby("itemid")["event_weight"]
    .sum()
    .sort_values(ascending=False)
)

weighted_popular_top10 = (
    weighted_popularity
    .head(10)
    .index
    .astype(int)
    .tolist()
)

weighted_popular_top10

[461686, 187946, 5411, 257040, 309778, 370653, 7943, 219512, 298009, 312728]

In [ ]:
popular_items_top10

[461686, 119736, 213834, 445351, 48030, 7943, 420960, 17478, 248455, 312728]

In [ ]:
def evaluate_weighted_popularity_user(user_id, k=10):
    recommended_items = set(
        weighted_popular_top10[:k]
    )

    actual_items = set(
        test_data.loc[
            test_data["visitorid"] == user_id,
            "itemid"
        ].astype(int)
    )

    hits = recommended_items & actual_items

    precision = (
        len(hits) / len(recommended_items)
        if recommended_items
        else 0
    )

    recall = (
        len(hits) / len(actual_items)
        if actual_items
        else 0
    )

    return {
        "visitorid": user_id,
        "hits": len(hits),
        "precision@10": precision,
        "recall@10": recall
    }

In [ ]:
random_new_users = segment_samples_random["new_user"]

weighted_pop_new_results = []

for user_id in random_new_users:
    weighted_pop_new_results.append(
        evaluate_weighted_popularity_user(user_id)
    )

weighted_pop_new_results_df = pd.DataFrame(
    weighted_pop_new_results
)

In [ ]:
weighted_pop_new_results_df[
    ["precision@10", "recall@10", "hits"]
].mean()

precision@10    0.001
recall@10       0.005
hits            0.010
dtype: float64

In [ ]:
(weighted_pop_new_results_df["hits"] > 0).sum()

np.int64(1)

In [ ]:
recommend_for_user_final(
    999999999,
    k=10
)

,itemid,score,source
0,461686,3350.0,weighted_popularity_cold_start
1,187946,2700.0,weighted_popularity_cold_start
2,5411,2192.0,weighted_popularity_cold_start
3,257040,1928.0,weighted_popularity_cold_start
4,309778,1779.0,weighted_popularity_cold_start
5,370653,1735.0,weighted_popularity_cold_start
6,7943,1674.0,weighted_popularity_cold_start
7,219512,1663.0,weighted_popularity_cold_start
8,298009,1538.0,weighted_popularity_cold_start
9,312728,1503.0,weighted_popularity_cold_start


In [ ]:
recommend_for_user_final(
    999999999,
    k=10
)

,itemid,score,source
0,461686,3350.0,weighted_popularity_cold_start
1,187946,2700.0,weighted_popularity_cold_start
2,5411,2192.0,weighted_popularity_cold_start
3,257040,1928.0,weighted_popularity_cold_start
4,309778,1779.0,weighted_popularity_cold_start
5,370653,1735.0,weighted_popularity_cold_start
6,7943,1674.0,weighted_popularity_cold_start
7,219512,1663.0,weighted_popularity_cold_start
8,298009,1538.0,weighted_popularity_cold_start
9,312728,1503.0,weighted_popularity_cold_start


In [ ]:
final_segment_samples = {}

for segment in ["new_user", "low_history", "established"]:
    segment_users = routing_segments_df[
        routing_segments_df["segment"] == segment
    ]

    final_segment_samples[segment] = (
        segment_users["visitorid"]
        .sample(
            n=500,
            random_state=42
        )
        .tolist()
    )

In [ ]:
{
    segment: len(users)
    for segment, users in final_segment_samples.items()
}

{'new_user': 500, 'low_history': 500, 'established': 500}

In [ ]:
import time

for segment in ["new_user", "low_history", "established"]:
    user_id = final_segment_samples[segment][0]

    start = time.time()
    evaluate_final_user(user_id)
    elapsed = time.time() - start

    print(segment, elapsed)

new_user 0.007454872131347656
low_history 0.029365062713623047
established 2.6145131587982178


In [ ]:
content_neighbor_cache = {}

def get_content_neighbors(item_id, k=20):
    item_id = int(item_id)
    key = (item_id, k)

    if key not in content_neighbor_cache:
        content_neighbor_cache[key] = recommend_similar_items(
            item_id,
            k=k
        )

    return content_neighbor_cache[key]

In [ ]:
content_neighbor_cache = {}

def get_content_neighbors(item_id, k=20):
    item_id = int(item_id)
    key = (item_id, k)

    if key not in content_neighbor_cache:
        content_neighbor_cache[key] = recommend_similar_items(
            item_id,
            k=k
        )

    return content_neighbor_cache[key]

In [ ]:
for segment in ["new_user", "low_history", "established"]:
    user_id = final_segment_samples[segment][0]

    start = time.time()
    evaluate_final_user(user_id)
    elapsed = time.time() - start

    print(segment, elapsed)

new_user 0.04004502296447754
low_history 0.022570133209228516
established 2.268333911895752


In [ ]:
for segment in ["new_user", "low_history", "established"]:
    user_id = final_segment_samples[segment][0]

    start = time.time()
    evaluate_final_user(user_id)
    elapsed = time.time() - start

    print(segment, elapsed)

new_user 0.012800216674804688
low_history 0.023493051528930664
established 2.2588820457458496


In [ ]:
len(content_neighbor_cache)

0

In [ ]:
len(content_neighbor_cache)

0

In [ ]:
content_neighbor_cache.clear()
len(content_neighbor_cache)

0

In [ ]:
user_id = final_segment_samples["established"][0]

start = time.time()
evaluate_final_user(user_id)
print("first run:", time.time() - start)

first run: 4.035331964492798


In [ ]:
len(content_neighbor_cache)

4

In [ ]:
start = time.time()
evaluate_final_user(user_id)
print("second run:", time.time() - start)

second run: 0.07253479957580566


In [ ]:
item_to_index = {
    int(item_id): idx
    for idx, item_id in enumerate(item_ids_full)
}

In [ ]:
def recommend_similar_items(item_id, k=10):
    target_index = item_to_index.get(int(item_id))

    if target_index is None:
        return pd.DataFrame(
            columns=["itemid", "similarity"]
        )

    similarities = cosine_similarity(
        full_item_feature_matrix[target_index],
        full_item_feature_matrix
    ).flatten()

    sorted_indices = similarities.argsort()[::-1]

    sorted_indices = sorted_indices[
        sorted_indices != target_index
    ]

    top_indices = sorted_indices[:k]

    return pd.DataFrame({
        "itemid": item_ids_full[top_indices],
        "similarity": similarities[top_indices]
    }).reset_index(drop=True)

In [ ]:
from sklearn.preprocessing import normalize

normalized_item_matrix = normalize(
    full_item_feature_matrix.astype("float32"),
    norm="l2",
    axis=1,
    copy=True
)

normalized_item_matrix.shape

(417053, 161379)

In [ ]:
content_neighbor_cache.clear()

In [ ]:
len(content_neighbor_cache)

0

In [ ]:
test_item = 460429

In [ ]:
start = time.time()

result = get_content_neighbors(
    test_item,
    k=20
)

print("First lookup:", time.time() - start)

result.head()

First lookup: 0.7803740501403809


,itemid,similarity
0,100656,0.916667
1,440711,0.893819
2,402554,0.875000
3,367270,0.875000
4,238891,0.875000


In [ ]:
start = time.time()

result = get_content_neighbors(
    test_item,
    k=20
)

print("Cached lookup:", time.time() - start)

Cached lookup: 9.489059448242188e-05


In [ ]:
len(content_neighbor_cache)

1

In [ ]:
validation_low_items = set()

for user_id in final_segment_samples["low_history"]:
    user_history = user_item_strength[
        user_item_strength["visitorid"] == user_id
    ]

    validation_low_items.update(
        user_history["itemid"].astype(int)
    )

len(validation_low_items)

606

In [ ]:
validation_established_items = set()

for user_id in final_segment_samples["established"]:
    user_history = (
        user_item_strength[
            user_item_strength["visitorid"] == user_id
        ]
        .sort_values("weight", ascending=False)
        .head(20)
    )

    validation_established_items.update(
        user_history["itemid"].astype(int)
    )

len(validation_established_items)

3298

In [ ]:
len(validation_low_items | validation_established_items)

3848

In [ ]:
validation_source_items = (
    validation_low_items
    | validation_established_items
)

len(validation_source_items)

3848

In [ ]:
validation_source_items = [
    item_id
    for item_id in validation_source_items
    if item_id in item_to_index
]

len(validation_source_items)

3619

In [ ]:
import time
import numpy as np
import pandas as pd

def warm_content_cache(
    item_ids,
    k=20,
    batch_size=16
):
    start_time = time.time()

    item_ids = [
        int(item_id)
        for item_id in item_ids
        if (int(item_id), k) not in content_neighbor_cache
        and int(item_id) in item_to_index
    ]

    print("Items to compute:", len(item_ids))

    for start_idx in range(0, len(item_ids), batch_size):
        batch_items = item_ids[
            start_idx:start_idx + batch_size
        ]

        batch_indices = [
            item_to_index[item_id]
            for item_id in batch_items
        ]

        similarities = (
            normalized_item_matrix[batch_indices]
            @ normalized_item_matrix.T
        ).toarray()

        for row_idx, (item_id, target_index) in enumerate(
            zip(batch_items, batch_indices)
        ):
            scores = similarities[row_idx]

            scores[target_index] = -1

            top_indices = np.argpartition(
                scores,
                -k
            )[-k:]

            top_indices = top_indices[
                np.argsort(scores[top_indices])[::-1]
            ]

            content_neighbor_cache[
                (item_id, k)
            ] = pd.DataFrame({
                "itemid": item_ids_full[top_indices],
                "similarity": scores[top_indices]
            }).reset_index(drop=True)

        if start_idx % (batch_size * 10) == 0:
            print(
                f"{min(start_idx + batch_size, len(item_ids))}"
                f"/{len(item_ids)} cached"
            )

    print(
        "Cache warming time:",
        time.time() - start_time
    )

In [ ]:
warm_content_cache(
    validation_source_items,
    k=20,
    batch_size=16
)

Items to compute: 3619
16/3619 cached
176/3619 cached
336/3619 cached
496/3619 cached
656/3619 cached
816/3619 cached
976/3619 cached
1136/3619 cached
1296/3619 cached
1456/3619 cached
1616/3619 cached
1776/3619 cached
1936/3619 cached
2096/3619 cached
2256/3619 cached
2416/3619 cached
2576/3619 cached
2736/3619 cached
2896/3619 cached
3056/3619 cached
3216/3619 cached
3376/3619 cached
3536/3619 cached
Cache warming time: 69.05328798294067


In [ ]:
len(content_neighbor_cache)

3620

In [ ]:
import time

final_segment_results = []

for segment, users in final_segment_samples.items():
    start_time = time.time()

    user_results = []

    for user_id in users:
        user_results.append(
            evaluate_final_user(user_id)
        )

    results_df = pd.DataFrame(user_results)

    elapsed = time.time() - start_time

    final_segment_results.append({
        "segment": segment,
        "precision@10": results_df["precision@10"].mean(),
        "recall@10": results_df["recall@10"].mean(),
        "avg_hits": results_df["hits"].mean(),
        "users_with_hit": (results_df["hits"] > 0).sum(),
        "hit_rate": (results_df["hits"] > 0).mean(),
        "evaluation_time_sec": elapsed
    })

final_segment_comparison_df = pd.DataFrame(
    final_segment_results
)

final_segment_comparison_df

,segment,precision@10,recall@10,avg_hits,users_with_hit,hit_rate,evaluation_time_sec
0,new_user,0.0004,0.003000,0.004,2,0.004,0.481158
1,low_history,0.0022,0.016667,0.022,10,0.020,1.692842
2,established,0.0038,0.015367,0.038,18,0.036,3.128056


In [ ]:
from pathlib import Path
import pickle
import numpy as np
from scipy.sparse import save_npz

PROJECT_ROOT = Path.cwd().parent
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

ARTIFACT_DIR.mkdir(exist_ok=True)

ARTIFACT_DIR

PosixPath('/Users/krrishmujoo/Projects/AI-Recomender-project/artifacts')

In [ ]:
save_npz(
    ARTIFACT_DIR / "normalized_item_matrix.npz",
    normalized_item_matrix
)

np.save(
    ARTIFACT_DIR / "item_ids.npy",
    item_ids_full
)

In [ ]:
with open(
    ARTIFACT_DIR / "item_to_index.pkl",
    "wb"
) as f:
    pickle.dump(item_to_index, f)

In [ ]:
with open(
    ARTIFACT_DIR / "weighted_popularity.pkl",
    "wb"
) as f:
    pickle.dump(weighted_popularity, f)

In [ ]:
with open(
    ARTIFACT_DIR / "user_history_count.pkl",
    "wb"
) as f:
    pickle.dump(user_history_count, f)

In [ ]:
with open(
    ARTIFACT_DIR / "content_neighbors.pkl",
    "wb"
) as f:
    pickle.dump(content_neighbor_cache, f)

In [ ]:
user_item_strength.to_parquet(
    ARTIFACT_DIR / "user_item_strength.parquet",
    index=False
)

ArrowKeyError: A type extension with name pandas.period already defined

In [ ]:
interaction_weights = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5
}

In [ ]:
train_cf = train_data[
    ["visitorid", "itemid", "event"]
].copy()

train_cf["weight"] = train_cf["event"].map(
    interaction_weights
)

NameError: name 'train_data' is not defined

In [94]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

# 1. Reload raw events
events = pd.read_csv(
    DATA_DIR / "events.csv"
)

# 2. Remove exact duplicates
events_clean = events.drop_duplicates().copy()

# 3. Recreate datetime
events_clean["datetime"] = pd.to_datetime(
    events_clean["timestamp"],
    unit="ms"
)

# 4. Recreate train split
train_data = events_clean[
    events_clean["datetime"] < "2015-09-01"
].copy()

train_data.shape

(2451983, 6)

In [95]:
interaction_weights = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5
}

train_cf = train_data[
    ["visitorid", "itemid", "event"]
].copy()

train_cf["weight"] = train_cf["event"].map(
    interaction_weights
)

user_item_strength = (
    train_cf
    .groupby(["visitorid", "itemid"])["weight"]
    .sum()
    .reset_index()
)

user_item_strength.shape

(1905695, 3)

In [96]:
user_item_strength.to_pickle(
    ARTIFACT_DIR / "user_item_strength.pkl"
)

In [97]:
(ARTIFACT_DIR / "user_item_strength.pkl").exists()

True

In [98]:
for path in ARTIFACT_DIR.iterdir():
    print(
        path.name,
        round(path.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

item_ids.npy 3.18 MB
weighted_popularity.pkl 5.1 MB
content_neighbors.pkl 1.81 MB
item_to_index.pkl 3.74 MB
user_history_count.pkl 8.24 MB
normalized_item_matrix.npz 41.13 MB
user_item_strength.pkl 43.62 MB
